# <h1 align="center"><b>Applio</b></h1>
<p align="center">A high-quality, lightweight voice conversion web interface and voice AI tool.</p>

<p align="center">
  <a href="https://discord.gg/wY7gmqTyEV" target="_blank">☎️ Discord Support</a> •
  <a href="https://github.com/IAHispano/Applio-app" target="_blank">💻 GitHub</a> •
  <a href="https://github.com/IAHispano/Applio-app/blob/main/TERMS_OF_USE.md" target="_blank">📜 Terms of Use</a>
</p>

---

### **How to use on Google Colab:**
1. *(Optional)* Run **Step 1: Mount Google Drive** if you want your trained models and audio files to automatically save to your Google Drive.
2. Run **Step 2: Install Applio** to set up the environment, dependencies, and web application.
3. *(Optional)* Run **Step 3: Sync with Google Drive** to link your existing Google Drive models.
4. Run **Step 4: Start Applio Web UI** and click the generated public link to open the app!

### **Step 1: Mount Google Drive (Optional)**
Mount your Google Drive to keep your models, weights, and audio outputs saved across runtime sessions.

In [ ]:
# @title Mount Google Drive
from google.colab import drive

try:
    drive.mount("/content/drive")
    print("✅ Google Drive mounted successfully!")
except Exception as e:
    print(f"⚠️ Notice: Drive mount skipped or failed ({e}). You can still use Applio with temporary runtime storage.")

### **Step 2: Install Applio & Requirements**
Clones the Applio repository, configures the environment, installs dependencies, and prepares the standalone web server.

In [ ]:
# @title Setup runtime environment & install Applio
import os
import subprocess
from pathlib import Path
from IPython.display import clear_output

REPO_URL = "https://github.com/IAHispano/Applio-app.git"
REPO_NAME = "Applio"
REPO_PATH = Path(f"/content/{REPO_NAME}")

%cd /content

# Clone or pull repository
if not REPO_PATH.exists():
    print("📥 Cloning Applio repository...")
    !git clone --depth 1 {REPO_URL} {REPO_NAME}
else:
    print("🔄 Applio repository found, pulling updates...")
    %cd {REPO_PATH}
    !git pull

%cd {REPO_PATH}
clear_output()

print("📦 Installing system dependencies...")
!apt-get update -qq
!apt-get install -qq -y portaudio19-dev ffmpeg psmisc > /dev/null 2>&1

print("⚡ Installing Python dependencies with uv...")
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null 2>&1
!uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match --system
!uv pip install -q pyngrok jupyter-ui-poll --system

print("📥 Downloading core RVC prerequisites...")
!python core.py prerequisites --models --pretraineds-hifigan

print("🌐 Setting up Node.js & building Web Application...")
print("⬢ Installing Node.js 22 (required by pnpm 11 / Next.js 15)...")
!curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash - > /dev/null 2>&1
!sudo apt-get install -y nodejs > /dev/null 2>&1
!node --version
!npm install -g -q pnpm > /dev/null 2>&1
!pnpm install --frozen-lockfile > /dev/null 2>&1 || pnpm install > /dev/null 2>&1
!pnpm build

print("🔗 Installing Cloudflare & LocalTunnel utilities...")
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!npm install -g -q localtunnel > /dev/null 2>&1

clear_output()
print("✅ Installation complete! Proceed to the next cell to launch Applio.")

### **Step 3: Sync with Google Drive (Optional)**
Sync models and logs with your Google Drive backup folder (`MyDrive/ApplioBackup`).

In [ ]:
# @title Sync models with Google Drive
# @markdown Run this cell to automatically Save / Load models from your mounted Google Drive
from IPython.display import display, clear_output
from pathlib import Path
import os

LOGS_PATH = Path(f"/content/Applio/logs")
BACKUPS_PATH = Path("/content/drive/MyDrive/ApplioBackup")
non_bak_folders = ["mute", "reference", "zips", "mute_spin", "mute_spin-v2"]
non_bak_path = Path("/tmp/rvc_logs")

def press_button(button):
    button.disabled = True

def get_date(path: Path):
    from datetime import datetime
    return datetime.fromtimestamp(int(path.stat().st_mtime))

def get_size(path: Path):
    res = !du -shx --apparent-size "{path}" 2>/dev/null
    if res:
        return res[0].split("	")[0] + "B"
    return "0B"

def sync_folders(folder: Path, backup: Path):
    from ipywidgets import widgets
    from jupyter_ui_poll import ui_events
    from time import sleep

    local = widgets.VBox([
        widgets.Label(f"Local: logs/{folder.name}/"),
        widgets.Label(f"Size: {get_size(folder)}"),
        widgets.Label(f"Last modified: {get_date(folder)}")
    ])
    remote = widgets.VBox([
        widgets.Label(f"Remote: ApplioBackup/{backup.name}/"),
        widgets.Label(f"Size: {get_size(backup)}"),
        widgets.Label(f"Last modified: {get_date(backup)}")
    ])
    separator = widgets.VBox([
        widgets.Label("|||"),
        widgets.Label("|||"),
        widgets.Label("|||")
    ])
    radio = widgets.RadioButtons(
        options=["Save local model to drive", "Keep remote model"]
    )
    button = widgets.Button(
        description="Sync",
        icon="upload",
        tooltip="Sync model"
    )
    button.on_click(press_button)

    clear_output()
    print(f"Model '{folder.name}' exists both locally and in Google Drive:")
    display(widgets.Box([local, separator, remote]))
    display(radio)
    display(button)

    with ui_events() as poll:
        while not button.disabled:
            poll(10)
            sleep(0.1)

    if radio.value == "Save local model to drive":
        !rm -rf "{backup}"
        !cp -r "{folder}" "{backup}"
    else:
        !rm -rf "{folder}"
        !cp -r "{backup}" "{folder}"

if Path("/content/drive/MyDrive").exists():
    BACKUPS_PATH.mkdir(parents=True, exist_ok=True)
    non_bak_path.mkdir(parents=True, exist_ok=True)

    if not LOGS_PATH.is_symlink():
        for folder_name in non_bak_folders:
            folder = LOGS_PATH / folder_name
            backup = BACKUPS_PATH / folder_name
            if folder.exists():
                !mkdir -p "{non_bak_path}"
                !mv "{folder}" "{non_bak_path}/" 2>/dev/null
                !rm -rf "{folder}"
            folder = non_bak_path / folder_name
            if backup.exists() and backup.resolve() != folder.resolve():
                !rm -rf "{backup}"
            if folder.exists():
                !ln -s "{folder}" "{backup}" 2>/dev/null

        for model in LOGS_PATH.iterdir():
            if model.is_dir() and not model.is_symlink() and model.name != ".ipynb_checkpoints":
                backup = BACKUPS_PATH / model.name
                if backup.exists() and backup.is_dir():
                    sync_folders(model, backup)
                else:
                    !rm -rf "{backup}"
                    !cp -r "{model}" "{backup}"

        !rm -rf "{LOGS_PATH}"
        !ln -s "{BACKUPS_PATH}" "{LOGS_PATH}"
        clear_output()
        print("✅ Models and backups are synchronized with Google Drive!")
    else:
        clear_output()
        print("✅ Models are already synced with Google Drive!")
else:
    print("ℹ️ Google Drive is not mounted. Run Step 1 if you wish to sync with Drive.")

### **Step 4: Start Applio Web UI**
Starts the Express API backend, Next.js frontend, and creates a public tunnel link so you can use Applio online for free in your browser.

In [ ]:
# @title Start Applio Web UI
# @markdown Select your preferred tunneling service and run this cell.
tunnel_method = "Cloudflare (Recommended - Free, No Token)"  # @param ["Cloudflare (Recommended - Free, No Token)", "Ngrok (Requires Token)", "LocalTunnel (Free)"]
ngrok_token = ""  # @param {type:"string"}
enable_tensorboard = False  # @param {type:"boolean"}

import os
import re
import time
import subprocess
import urllib.request
from IPython.display import display, HTML, clear_output

%cd /content/Applio

# Terminate existing servers
!fuser -k 3000/tcp 2>/dev/null; fuser -k 8000/tcp 2>/dev/null; fuser -k 6006/tcp 2>/dev/null; true

if enable_tensorboard:
    %load_ext tensorboard
    %tensorboard --logdir logs --port 6006 --bind_all

print("🚀 Starting Applio backend engine...")
!API_PORT=8000 nohup node app/api/dist/index.js >/tmp/applio-api.log 2>&1 &

print("🌐 Starting Applio Next.js frontend...")
!PORT=3000 HOSTNAME="0.0.0.0" nohup node app/web/.next/standalone/server.js >/tmp/applio-web.log 2>&1 &

# Wait for readiness
print("⏳ Waiting for server readiness...")
ready = False
for _ in range(30):
    try:
        r = urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=1)
        if r.status == 200:
            ready = True
            break
    except Exception:
        time.sleep(1)

if not ready:
    print("⚠️ Warning: API health check timed out. Checking server logs:")
    !tail -n 15 /tmp/applio-api.log
    !tail -n 15 /tmp/applio-web.log

clear_output()

def show_card(url, tunnel_name, extra_info=""):
    html = f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
                background: #141414; border: 1px solid rgba(255, 255, 255, 0.12);
                border-radius: 14px; padding: 22px 26px; max-width: 540px; margin: 16px 0;
                box-shadow: 0 10px 30px rgba(0,0,0,0.5); color: #f5f5f5;">
        <div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 12px;">
            <div style="display: flex; align-items: center; gap: 8px;">
                <span style="font-size: 20px; font-weight: 700; letter-spacing: -0.02em;">Applio</span>
                <span style="font-size: 11px; font-weight: 500; color: #a3a3a3; background: rgba(255,255,255,0.08);
                             border: 1px solid rgba(255,255,255,0.1); border-radius: 4px; padding: 1px 6px;">Web UI</span>
            </div>
            <span style="font-size: 12px; color: #4ade80; font-weight: 500;">● Online ({tunnel_name})</span>
        </div>
        <p style="font-size: 13px; color: #a3a3a3; margin: 0 0 16px 0; line-height: 1.4;">
            Your Applio web instance is ready. Click the button below to access the full web application:
        </p>
        <a href="{url}" target="_blank"
           style="display: inline-block; background: #ffffff; color: #0a0a0a; font-size: 13px; font-weight: 600;
                  padding: 10px 22px; border-radius: 999px; text-decoration: none; box-shadow: 0 2px 8px rgba(0,0,0,0.2);">
            Open Applio Web App ↗
        </a>
        <div style="margin-top: 14px; font-size: 12px; color: #737373;">
            Public Link: <a href="{url}" target="_blank" style="color: #60a5fa; word-break: break-all;">{url}</a>
        </div>
        {extra_info}
    </div>
    """
    display(HTML(html))

if "Cloudflare" in tunnel_method:
    os.system("pkill -f cloudflared 2>/dev/null")
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:3000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    tunnel_url = None
    t0 = time.time()
    while time.time() - t0 < 30:
        line = proc.stdout.readline()
        if not line:
            time.sleep(0.5)
            continue
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            tunnel_url = match.group(0)
            break
    if tunnel_url:
        show_card(tunnel_url, "Cloudflare")
    else:
        print("❌ Could not obtain Cloudflare tunnel URL.")

elif "Ngrok" in tunnel_method:
    if not ngrok_token or "http" in ngrok_token or len(ngrok_token.strip()) < 10:
        print("❌ Please enter your Ngrok Authtoken in the cell settings.")
        print("Obtain your free token here: https://dashboard.ngrok.com/get-started/your-authtoken")
    else:
        from pyngrok import ngrok
        ngrok.kill()
        ngrok.set_auth_token(ngrok_token.strip())
        listener = ngrok.connect(3000)
        show_card(listener.public_url, "Ngrok")

elif "LocalTunnel" in tunnel_method:
    os.system("pkill -f 'lt --port' 2>/dev/null")
    endpoint_ip = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode("utf8").strip()
    proc = subprocess.Popen(["lt", "--port", "3000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tunnel_url = None
    t0 = time.time()
    while time.time() - t0 < 25:
        line = proc.stdout.readline()
        if "your url is:" in line:
            tunnel_url = line.split("your url is:")[-1].strip()
            break
    if tunnel_url:
        extra = f"""<div style="margin-top: 10px; font-size: 12px; color: #fbbf24;">
            ⚠️ <b>Tunnel Password IP:</b> <code style="background: rgba(255,255,255,0.1); padding: 2px 6px; border-radius: 4px;">{endpoint_ip}</code> (Enter this if prompted)
        </div>"""
        show_card(tunnel_url, "LocalTunnel", extra)
    else:
        print("❌ Could not obtain LocalTunnel URL.")

print("\nApplio server is running. Keep this cell active while using the application.")
print("To stop the server, click the interrupt/stop button in the cell toolbar.")
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nShutting down Applio server...")
    os.system("fuser -k 3000/tcp 2>/dev/null; fuser -k 8000/tcp 2>/dev/null")